# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene1407/ML-1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose Logistic Regression as my first model because the target, position_worsened, is binary. It provides a simple and interpretable starting point for understanding which March search-performance signals are associated with April ranking deterioration.

I will also test a Random Forest to check whether non-linear patterns improve the results. Since my lane focuses on prioritizing content for review, I will evaluate the ranked predictions using Precision@50 and Precision@100, alongside the Week 4 rule-based baseline. These results are decision-support signals, not proof that a page will lose ranking.

In [4]:
import pandas as pd

feature_df = pd.read_parquet(
    r"C:\Users\LENOVO\Downloads\ML-1\work\outputs\w03_feature_frame.parquet"
)

print("Feature frame shape:", feature_df.shape)
print("Columns:", feature_df.columns.tolist())

Feature frame shape: (158549, 8)
Columns: ['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'march_ctr', 'active_days', 'position_worsened']


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm the feature frame before modeling

print("Feature frame shape:", feature_df.shape)
print("\nColumns:")
print(feature_df.columns.tolist())

print("\nTarget distribution:")
print(feature_df["position_worsened"].value_counts())

print("\nTarget percentage:")
print(
    (feature_df["position_worsened"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nMissing values:")
print(feature_df.isnull().sum())
%whos

Feature frame shape: (158549, 8)

Columns:
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'march_ctr', 'active_days', 'position_worsened']

Target distribution:
position_worsened
1    99146
0    59403
Name: count, dtype: int64

Target percentage:
position_worsened
1    62.53
0    37.47
Name: proportion, dtype: float64

Missing values:
client_hash_id       0
content_hash_id      0
impressions          0
clicks               0
avg_position         0
march_ctr            0
active_days          0
position_worsened    0
dtype: int64
Variable     Type         Data/Info
-----------------------------------
feature_df   DataFrame    Shape: (158549, 8)
pd           module       <module 'pandas' from 'c:<...>es\\pandas\\__init__.py'>


In [6]:
# Step 4: Verify feature frame before modeling

print("Feature frame shape:", feature_df.shape)

print("\nColumns:")
print(feature_df.columns.tolist())

print("\nTarget distribution:")
print(feature_df["position_worsened"].value_counts())

print("\nTarget percentage:")
print(
    (feature_df["position_worsened"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\nMissing values:")
print(feature_df.isnull().sum())

print("\nDuplicate rows:", feature_df.duplicated().sum())

Feature frame shape: (158549, 8)

Columns:
['client_hash_id', 'content_hash_id', 'impressions', 'clicks', 'avg_position', 'march_ctr', 'active_days', 'position_worsened']

Target distribution:
position_worsened
1    99146
0    59403
Name: count, dtype: int64

Target percentage:
position_worsened
1    62.53
0    37.47
Name: proportion, dtype: float64

Missing values:
client_hash_id       0
content_hash_id      0
impressions          0
clicks               0
avg_position         0
march_ctr            0
active_days          0
position_worsened    0
dtype: int64

Duplicate rows: 0


In [8]:
# 1. Define features and target

import numpy as np
import pandas as pd

features = [
    "impressions",
    "clicks",
    "avg_position",
    "march_ctr",
    "active_days"
]

target = "position_worsened"

X = feature_df[features].copy()
y = feature_df[target].astype(int).copy()

print("Feature frame:", feature_df.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:", features)
print("\nTarget distribution:")
print(y.value_counts())
print("\nTarget percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

Feature frame: (158549, 8)
X shape: (158549, 5)
y shape: (158549,)

Features: ['impressions', 'clicks', 'avg_position', 'march_ctr', 'active_days']

Target distribution:
position_worsened
1    99146
0    59403
Name: count, dtype: int64

Target percentages:
position_worsened
1    62.53
0    37.47
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a grouped train-test split based on client_hash_id. This ensures that records belonging to the same client do not appear in both training and testing data.

This split evaluates whether the models can identify ranking deterioration for clients they have not seen during training. I used a fixed random seed for reproducibility. The client and content identifiers are used only for splitting and analysis, not as model features.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 2. Create a client-grouped train-test split

from sklearn.model_selection import GroupShuffleSplit

groups = feature_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("Overlapping clients:", len(train_clients.intersection(test_clients)))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(4))

assert len(train_clients.intersection(test_clients)) == 0
assert len(X_train) + len(X_test) == len(feature_df)

print("\nGrouped split validation passed.")

Training rows: 137445
Testing rows: 21104
Training clients: 36
Testing clients: 10
Overlapping clients: 0

Training target distribution:
position_worsened
1    0.6026
0    0.3974
Name: proportion, dtype: float64

Testing target distribution:
position_worsened
1    0.7731
0    0.2269
Name: proportion, dtype: float64

Grouped split validation passed.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I trained Logistic Regression and Random Forest using the same training data and evaluated them on the same held-out clients.

I also reconstructed the Week 4 rule-based baseline using March-only signals. All three approaches are evaluated on the same test records using Precision@50 and Precision@100. I report the test-set positive rate as a reference point.

This comparison helps determine whether the models improve the prioritization of content compared with the simpler rule-based approach.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 3A. Reconstruct the Week 4 rule-based baseline
# March-only features are used. No target information is used.

baseline_df = feature_df[
    ["client_hash_id", "content_hash_id"] + features
].copy()

# Group average positions into interpretable buckets
position_bins = [0, 3, 10, 20, 50, np.inf]

position_labels = [
    "0-3",
    "3-10",
    "10-20",
    "20-50",
    "50+"
]

baseline_df["position_bucket"] = pd.cut(
    baseline_df["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

# Calculate median CTR for each position bucket
position_medians = baseline_df.groupby(
    "position_bucket",
    observed=False
)["march_ctr"].median()

baseline_df["expected_ctr"] = (
    baseline_df["position_bucket"].map(position_medians).astype(float)
)

# Identify high-visibility content
impression_threshold = baseline_df["impressions"].quantile(0.75)

baseline_df["high_visibility"] = (
    baseline_df["impressions"] >= impression_threshold
)

# Calculate positive CTR gap
baseline_df["ctr_gap"] = (
    baseline_df["expected_ctr"] - baseline_df["march_ctr"]
).clip(lower=0)

# Baseline score
baseline_df["baseline_score"] = (
    baseline_df["impressions"] * baseline_df["ctr_gap"]
)

# Apply the rule
baseline_df["baseline_score"] = np.where(
    baseline_df["high_visibility"],
    baseline_df["baseline_score"],
    0
)

baseline_df["reason_code"] = np.where(
    (baseline_df["high_visibility"]) &
    (baseline_df["ctr_gap"] > 0),
    "high_visibility_low_ctr",
    "no_priority_signal"
)

print("Impression threshold:", impression_threshold)
print("\nPosition-group median CTR:")
print(position_medians)

print("\nBaseline score summary:")
print(baseline_df["baseline_score"].describe())

print("\nReason-code distribution:")
print(baseline_df["reason_code"].value_counts())

Impression threshold: 1264.0

Position-group median CTR:
position_bucket
0-3      0.067814
3-10     0.027590
10-20    0.000000
20-50    0.000000
50+      0.000000
Name: march_ctr, dtype: float64

Baseline score summary:
count    158549.000000
mean          1.692222
std          51.783766
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max       12003.934135
Name: baseline_score, dtype: float64

Reason-code distribution:
reason_code
no_priority_signal         156394
high_visibility_low_ctr      2155
Name: count, dtype: int64


In [11]:
# 3B. Train Logistic Regression and Random Forest

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

# Random Forest
random_forest_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=15,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

print("Logistic Regression training completed.")
print("Random Forest training completed.")

Logistic Regression training completed.
Random Forest training completed.


In [13]:
# 3C. Generate test-set probabilities

lr_probabilities = logistic_model.predict_proba(X_test)[:, 1]

rf_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

# Match baseline scores to the exact test rows
baseline_test_scores = baseline_df.iloc[test_idx][
    "baseline_score"
].to_numpy()

print("Logistic Regression predictions:", len(lr_probabilities))
print("Random Forest predictions:", len(rf_probabilities))
print("Baseline scores:", len(baseline_test_scores))

assert len(lr_probabilities) == len(y_test)
assert len(rf_probabilities) == len(y_test)
assert len(baseline_test_scores) == len(y_test)

print("\nAll predictions are aligned with the test set.")

Logistic Regression predictions: 21104
Random Forest predictions: 21104
Baseline scores: 21104

All predictions are aligned with the test set.


In [14]:
# 3C. Generate test-set probabilities

lr_probabilities = logistic_model.predict_proba(X_test)[:, 1]

rf_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

# Match baseline scores to the exact test rows
baseline_test_scores = baseline_df.iloc[test_idx][
    "baseline_score"
].to_numpy()

print("Logistic Regression predictions:", len(lr_probabilities))
print("Random Forest predictions:", len(rf_probabilities))
print("Baseline scores:", len(baseline_test_scores))

assert len(lr_probabilities) == len(y_test)
assert len(rf_probabilities) == len(y_test)
assert len(baseline_test_scores) == len(y_test)

print("\nAll predictions are aligned with the test set.")

Logistic Regression predictions: 21104
Random Forest predictions: 21104
Baseline scores: 21104

All predictions are aligned with the test set.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I examined incorrect predictions from both models to understand where the models struggle. I also inspected feature importance to identify which March performance signals contributed most to the Random Forest predictions.

Feature importance indicates predictive association within this dataset. It does not establish causation. The results should be interpreted as exploratory decision-support signals.

In [17]:
# Generate class predictions from existing probabilities

lr_predictions = (lr_probabilities >= 0.5).astype(int)
rf_predictions = (rf_probabilities >= 0.5).astype(int)

print("Logistic Regression predictions:", len(lr_predictions))
print("Random Forest predictions:", len(rf_predictions))

print("\nPredictions generated successfully!")

Logistic Regression predictions: 21104
Random Forest predictions: 21104

Predictions generated successfully!


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 4A. Identify incorrect predictions

error_df = feature_df.iloc[test_idx][
    ["client_hash_id", "content_hash_id"] + features + [target]
].copy()

error_df["lr_prediction"] = lr_predictions
error_df["rf_prediction"] = rf_predictions

error_df["lr_probability"] = lr_probabilities
error_df["rf_probability"] = rf_probabilities

error_df["lr_error"] = (
    error_df[target] != error_df["lr_prediction"]
)

error_df["rf_error"] = (
    error_df[target] != error_df["rf_prediction"]
)

print("Logistic Regression errors:", error_df["lr_error"].sum())
print("Random Forest errors:", error_df["rf_error"].sum())

print("\nLogistic Regression error rate:")
print(round(error_df["lr_error"].mean(), 4))

print("\nRandom Forest error rate:")
print(round(error_df["rf_error"].mean(), 4))

Logistic Regression errors: 5151
Random Forest errors: 8604

Logistic Regression error rate:
0.2441

Random Forest error rate:
0.4077


In [19]:
# 4B. Inspect up to three Random Forest errors

wrong_cases = error_df[
    error_df["rf_error"]
].copy()

wrong_cases["confidence"] = np.maximum(
    wrong_cases["rf_probability"],
    1 - wrong_cases["rf_probability"]
)

wrong_cases = wrong_cases.sort_values(
    "confidence",
    ascending=False
)

review_columns = [
    "impressions",
    "clicks",
    "avg_position",
    "march_ctr",
    "active_days",
    "position_worsened",
    "rf_prediction",
    "rf_probability"
]

display(
    wrong_cases[review_columns].head(3).round(4)
)

,impressions,clicks,avg_position,march_ctr,active_days,position_worsened,rf_prediction,rf_probability
51636,7754.0,22.0,1.4300,0.2837,27,0,1,0.9533
143676,206.0,1.0,1.4272,0.4854,20,0,1,0.9306
20307,12872.0,20.0,44.7671,0.1554,31,1,0,0.0708


In [20]:
# 4C. Inspect Random Forest feature importance

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": random_forest_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

display(importance_df.round(4))

,Feature,Importance
2,avg_position,0.6654
0,impressions,0.1526
3,march_ctr,0.0787
4,active_days,0.0650
1,clicks,0.0382


In [21]:
# 4D. Permutation importance on a manageable test sample

from sklearn.inspection import permutation_importance

sample_size = min(5000, len(X_test))

X_importance = X_test.sample(
    n=sample_size,
    random_state=42
)

y_importance = y_test.loc[X_importance.index]

permutation_result = permutation_importance(
    random_forest_model,
    X_importance,
    y_importance,
    n_repeats=5,
    random_state=42,
    scoring="average_precision",
    n_jobs=-1
)

permutation_df = pd.DataFrame({
    "Feature": features,
    "Importance_Mean": permutation_result.importances_mean,
    "Importance_Std": permutation_result.importances_std
}).sort_values(
    "Importance_Mean",
    ascending=False
)

display(permutation_df.round(4))

,Feature,Importance_Mean,Importance_Std
2,avg_position,0.0823,0.0053
0,impressions,0.0095,0.0024
4,active_days,0.0031,0.0015
3,march_ctr,0.0030,0.0013
1,clicks,0.0000,0.0007


In [22]:
# 4E. Compare model errors

error_summary = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Incorrect Predictions": [
        error_df["lr_error"].sum(),
        error_df["rf_error"].sum()
    ],
    "Error Rate": [
        error_df["lr_error"].mean(),
        error_df["rf_error"].mean()
    ]
})

display(error_summary.round(4))

,Model,Incorrect Predictions,Error Rate
0,Logistic Regression,5151,0.2441
1,Random Forest,8604,0.4077


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

The notebook uses March 2026 features to predict the April 2026 observed outcome. Client and content identifiers are excluded from model training. The train-test split is grouped by client, and the same held-out records are used to compare the baseline and both models.

The evaluation includes ranking metrics and classification metrics. Feature importance is treated as descriptive rather than causal. The results are exploratory and intended to support content-review prioritization.

In [24]:
required = [
    "train_clients",
    "test_clients",
    "baseline_test_scores",
    "comparison_df",
    "error_summary",
    "importance_df"
]

for name in required:
    print(f"{name}: {name in globals()}")

train_clients: True
test_clients: True
baseline_test_scores: True
comparison_df: False
error_summary: True
importance_df: True


In [26]:
# Recreate the comparison table

from sklearn.metrics import precision_score

def precision_at_k(y_true, scores, k):
    top_indices = scores.argsort()[::-1][:k]
    return y_true.iloc[top_indices].sum() / len(top_indices)

comparison_df = pd.DataFrame({
    "Method": [
        "Week 4 Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "Precision@50": [
        precision_at_k(y_test.reset_index(drop=True),
                        np.asarray(baseline_test_scores), 50),
        precision_at_k(y_test.reset_index(drop=True),
                        np.asarray(lr_probabilities), 50),
        precision_at_k(y_test.reset_index(drop=True),
                        np.asarray(rf_probabilities), 50)
    ],
    "Precision@100": [
        precision_at_k(y_test.reset_index(drop=True),
                        np.asarray(baseline_test_scores), 100),
        precision_at_k(y_test.reset_index(drop=True),
                        np.asarray(lr_probabilities), 100),
        precision_at_k(y_test.reset_index(drop=True),
                        np.asarray(rf_probabilities), 100)
    ]
})

display(comparison_df)

,Method,Precision@50,Precision@100
0,Week 4 Baseline,0.86,0.92
1,Logistic Regression,0.94,0.96
2,Random Forest,0.90,0.93


In [27]:
# Final self-check

print("1. Feature frame available:", "feature_df" in globals())
print("2. Expected feature columns:", all(f in feature_df.columns for f in features))
print("3. Target available:", target in feature_df.columns)
print("4. No missing feature values:", not X.isnull().any().any())
print("5. No missing target values:", not y.isnull().any())
print("6. Client overlap:", len(train_clients.intersection(test_clients)))
print("7. Baseline and model predictions aligned:",
      len(baseline_test_scores) == len(lr_probabilities) == len(rf_probabilities))
print("8. Comparison table created:", not comparison_df.empty)
print("9. Error analysis created:", not error_summary.empty)
print("10. Feature importance created:", not importance_df.empty)

assert len(train_clients.intersection(test_clients)) == 0
assert len(baseline_test_scores) == len(y_test)
assert len(lr_probabilities) == len(y_test)
assert len(rf_probabilities) == len(y_test)

print("\nML-08 self-check completed.")

1. Feature frame available: True
2. Expected feature columns: True
3. Target available: True
4. No missing feature values: True
5. No missing target values: True
6. Client overlap: 0
7. Baseline and model predictions aligned: True
8. Comparison table created: True
9. Error analysis created: True
10. Feature importance created: True

ML-08 self-check completed.
